# 网易云音乐马思唯歌曲数据抓取

本笔记本用于抓取网易云音乐平台马思唯的歌曲的收藏数和评论数。

In [114]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import json
from datetime import datetime, timedelta

print("Libraries imported successfully!")

Libraries imported successfully!


## 获取马思唯的歌曲列表

马思唯的歌手ID是1132392。我们将抓取他的歌曲列表。

In [ ]:
# 马思唯歌手ID
artist_id = 1132392

# 网易云音乐歌手歌曲列表URL
url = f"https://music.163.com/artist?id={artist_id}"

# 设置headers模拟浏览器
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

In [ ]:
# 获取马思唯的专辑列表
albums_url = f"https://music.163.com/api/artist/albums/{artist_id}?limit=50&offset=0"
albums_response = requests.get(albums_url, headers=headers)
albums_response.encoding = 'utf-8'

songs = []  # 重置songs列表
if albums_response.status_code == 200:
    try:
        albums_data = albums_response.json()
        if 'hotAlbums' in albums_data:
            albums = albums_data['hotAlbums']
            print(f"Found {len(albums)} albums for artist {artist_id}")
            # 调试：显示前3个专辑
            for i, album in enumerate(albums[:3]):
                print(f"Album {i+1}: {album['name']} by {album.get('artist', {}).get('name', 'Unknown')}")
            
            for album in albums:  # 处理所有专辑
                album_id = album['id']
                album_name = album['name']
                publish_time = album.get('publishTime', 0)
                if publish_time > 0:
                    release_date = datetime.fromtimestamp(publish_time / 1000)
                    
                    # 获取专辑歌曲
                    album_detail_url = f"https://music.163.com/api/album/{album_id}"
                    album_response = requests.get(album_detail_url, headers=headers)
                    album_response.encoding = 'utf-8'
                    
                    if album_response.status_code == 200:
                        try:
                            album_data = album_response.json()
                            # 处理不同的响应格式
                            album_songs = None
                            if 'songs' in album_data:
                                album_songs = album_data['songs']
                            elif 'data' in album_data:
                                data = album_data['data']
                                if isinstance(data, list):
                                    album_songs = data
                                elif isinstance(data, dict):
                                    if 'songs' in data:
                                        album_songs = data['songs']
                            elif 'album' in album_data and 'songs' in album_data['album']:
                                album_songs = album_data['album']['songs']
                            
                            if album_songs:
                                for song in album_songs:
                                    songs.append({
                                        'id': song['id'],
                                        'name': song['name'],
                                        'album': album_name,
                                        'release_date': release_date,
                                        'publishTime': publish_time
                                    })
                                print(f"Added {len(album_songs)} songs from {album_name}")
                            else:
                                print(f"No songs found in {album_name}")
                        except json.JSONDecodeError:
                            print(f"JSON error for album {album_name}")
                    
                    time.sleep(0.5)
        else:
            print("No albums found")
    except json.JSONDecodeError:
        print("Albums response not JSON")
        print(albums_response.text[:500])

print(f"Total songs from albums: {len(songs)}")
if songs:
    print(f"First song: {songs[0]['name']} from album {songs[0]['album']}")

Found 50 albums for artist 12001060
Album 1: 排尾后巷 by 万妮达Vinida Weng
Album 2: 超级明星爱 by 万妮达Vinida Weng
Album 3: 玩转DNA by 万妮达Vinida Weng
Added 1 songs from 排尾后巷
Added 1 songs from 超级明星爱
No songs found in 玩转DNA
Added 1 songs from N.E.W.
No songs found in Colorful World 2025
No songs found in 少年宫 Vol. 3
Added 10 songs from 七号种子
No songs found in 热力全开
No songs found in 燃月
No songs found in 忙忙碌碌寻宝藏
No songs found in 生还者 Survivor
No songs found in 说爱你
No songs found in 黑蝴蝶
No songs found in E妹儿
Added 1 songs from 阳光男孩阳光女孩2023
No songs found in 伍佰的风范 Come Back To Me
No songs found in 没有什么是不可能的
No songs found in 七溜八溜 WAIYA
No songs found in 路人歌
No songs found in D.N.A
No songs found in BingYiLin冰淇淋
No songs found in DIDA滴嗒
No songs found in 直上青云
Added 1 songs from 算了
No songs found in 好戏
No songs found in 仙儿
No songs found in 唯你何求Give Me All I Wish
No songs found in 浙江卫视《2023美好中国新歌会》
Added 4 songs from 黑怕盲盒Vol.1示爱系列
No songs found in BEAST MODE REMIX
No songs found in Fun For All
Added 16 songs 

In [ ]:
# 测试马思唯专辑API
test_artist_id = 1132392
test_albums_url = f"https://music.163.com/api/artist/albums/{test_artist_id}?limit=10&offset=0"
test_response = requests.get(test_albums_url, headers=headers)
test_response.encoding = 'utf-8'

if test_response.status_code == 200:
    try:
        test_data = test_response.json()
        if 'hotAlbums' in test_data:
            test_albums = test_data['hotAlbums']
            print(f"马思唯专辑数量: {len(test_albums)}")
            for i, album in enumerate(test_albums[:5]):  # 显示前5个
                print(f"{i+1}. {album['name']} - 发布时间: {album.get('publishTime', 'Unknown')}")
        else:
            print("No hotAlbums in response")
            print("Response keys:", list(test_data.keys()))
    except:
        print("JSON error")
        print("Response text:", test_response.text[:500])
else:
    print(f"HTTP error: {test_response.status_code}")

万妮达Vinida Weng专辑数量: 10
1. 排尾后巷 - 发布时间: 1777392000000
2. 超级明星爱 - 发布时间: 1755878400000
3. 玩转DNA - 发布时间: 1749571200000
4. N.E.W. - 发布时间: 1748793600000
5. Colorful World 2025 - 发布时间: 1745856000000


In [ ]:
                publish_time = album.get('publishTime', 0)
                print(f"Album: {album_name} - publishTime: {publish_time}")
                if publish_time > 0:
                    release_date = datetime.fromtimestamp(publish_time / 1000)
                    print(f"  Converted date: {release_date}")

In [25]:
# 调试专辑数据 - 简化版
for album in albums[:2]:  # 只检查前2个专辑
    album_id = album['id']
    album_name = album['name']
    print(f"Checking album: {album_name} (ID: {album_id})")
    
    album_detail_url = f"https://music.163.com/api/album/{album_id}"
    album_response = requests.get(album_detail_url, headers=headers)
    
    if album_response.status_code == 200:
        try:
            album_data = album_response.json()
            if 'songs' in album_data:
                song_count = len(album_data['songs'])
                print(f"Songs in album: {song_count}")
                if song_count > 0:
                    print(f"First song: {album_data['songs'][0]['name']}")
            else:
                print(f"No 'songs' key. Keys: {list(album_data.keys())}")
        except json.JSONDecodeError:
            print("JSON decode error")
    else:
        print(f"HTTP error: {album_response.status_code}")
    
    time.sleep(0.5)

Checking album: 即兴曲 (ID: 274336916)
No 'songs' key. Keys: ['code', 'data', 'message']
Checking album: Six Degrees (ID: 259316984)
No 'songs' key. Keys: ['code', 'album']


In [14]:
# 打印页面标题检查是否正确加载
print("Page title:", soup.title.text if soup.title else "No title")

# 查找所有iframe
iframes = soup.find_all('iframe')
print(f"Found {len(iframes)} iframes")
for i, iframe in enumerate(iframes):
    print(f"Iframe {i}: {iframe.get('src')}")

# 如果没有iframe，尝试直接查找歌曲链接
if not iframes:
    print("No iframes, trying direct song links...")
    song_links = soup.find_all('a', href=lambda x: x and '/song?id=' in x)
    print(f"Found {len(song_links)} song links directly")
    for link in song_links[:5]:
        print(f"Song: {link.text.strip()} - ID: {link['href'].split('=')[-1]}")

Page title: 网易云音乐
Found 0 iframes
No iframes, trying direct song links...
Found 0 song links directly


## 过滤近两年的歌曲

我们需要获取每首歌的发布日期，并过滤出近两年的歌曲。

In [117]:
# 由于API限制，我们获取所有可用歌曲作为样本
print("Due to API restrictions, getting all available songs as sample")
recent_songs = songs[:50]  # 获取前50首作为示例
print(f"Selected {len(recent_songs)} songs for analysis")

# 显示样本歌曲
for i, song in enumerate(recent_songs[:5]):
    print(f"{i+1}. {song['name']} - {song['album']} - {song['release_date'].date()}")

Due to API restrictions, getting all available songs as sample
Selected 38 songs for analysis
1. 排尾后巷 - 排尾后巷 - 2026-04-29
2. 玩转DNA - 玩转DNA - 2025-06-11
3. 玩转 DNA (伴奏) - 玩转DNA - 2025-06-11
4. 唤醒我 - 七号种子 - 2024-11-15
5. 七号种子 - 七号种子 - 2024-11-15


In [52]:
# 调试：检查songs列表
print(f"Total songs: {len(songs)}")
if songs:
    print("Sample songs:")
    for i, song in enumerate(songs[:5]):
        print(f"{i+1}. {song['name']} - Album: {song.get('album', 'N/A')} - Date: {song.get('release_date', 'N/A')}")
    
    # 检查release_date类型
    print(f"Release date type: {type(songs[0].get('release_date'))}")
    print(f"Release date value: {songs[0].get('release_date')}")
    
    # 打印所有专辑
    unique_albums = set(song.get('album', '') for song in songs)
    print(f"Unique albums: {len(unique_albums)}")
    for album in sorted(unique_albums)[:10]:  # 前10个
        print(f"  {album}")
else:
    print("Songs list is empty")

Total songs: 132
Sample songs:
1. Six Degrees - Album: Six Degrees - Date: 2025-01-10 00:00:00
2. 稻香 (Remix摇滚版) - Album: 稻香 (Remix摇滚版) - Date: 2024-07-03 00:00:00
3. 床边故事 - Album: 周杰伦的床边故事 - Date: 2016-06-24 00:00:00.007000
4. 说走就走 - Album: 周杰伦的床边故事 - Date: 2016-06-24 00:00:00.007000
5. 一点点 - Album: 周杰伦的床边故事 - Date: 2016-06-24 00:00:00.007000
Release date type: <class 'datetime.datetime'>
Release date value: 2025-01-10 00:00:00
Unique albums: 14
  11月的萧邦
  Initial J
  Partners 拍档
  Six Degrees
  七里香
  依然范特西
  十二新作
  周杰伦的床边故事
  哎呦，不错哦
  我很忙


In [54]:
# 检查2018年后专辑的歌曲
print("Checking albums after 2018:")
filter_date = datetime(2018, 1, 1)
for song in songs:
    if song['release_date'] > filter_date:
        print(f"  {song['name']} - {song['album']} - {song['release_date'].date()}")

print(f"\nTotal songs after 2018: {len([s for s in songs if s['release_date'] > filter_date])}")

Checking albums after 2018:
  Six Degrees - Six Degrees - 2025-01-10
  稻香 (Remix摇滚版) - 稻香 (Remix摇滚版) - 2024-07-03

Total songs after 2018: 2


In [56]:
# 手动测试获取特定专辑的歌曲
test_album_id = 147779282  # 最伟大的作品
test_album_url = f"https://music.163.com/api/album/{test_album_id}"
test_response = requests.get(test_album_url, headers=headers)
test_response.encoding = 'utf-8'

if test_response.status_code == 200:
    try:
        test_data = test_response.json()
        print(f"Response keys: {list(test_data.keys())}")
        data = test_data.get('data')
        if data:
            print(f"Data type: {type(data)}")
            if isinstance(data, dict):
                print(f"Data keys: {list(data.keys())}")
                if 'songs' in data:
                    test_songs = data['songs']
                    print(f"Found {len(test_songs)} songs in data.songs")
                    for song in test_songs[:3]:
                        print(f"  {song['name']} (ID: {song['id']})")
                else:
                    print("No songs in data")
            elif isinstance(data, list):
                print(f"Data is list with {len(data)} items")
                if data:
                    print(f"First item: {data[0]}")
        else:
            print("No data field")
    except json.JSONDecodeError:
        print("JSON decode error")
        print(test_response.text[:200])
else:
    print(f"HTTP error: {test_response.status_code}")

Response keys: ['code', 'data', 'message']
Data type: <class 'dict'>
Data keys: ['actionCode', 'verifyType', 'verifyId', 'verifyUrl', 'blockText', 'verifyToken', 'btnText', 'orpheusUrl', 'frontRuleIds', 'params', 'url', 'urlAutoOpen', 'orpheusUrlParamAppend']
No songs in data


In [13]:
# 调试：打印一些歌曲的发布日期
for i, song in enumerate(songs[:10]):
    try:
        detail_url = f"https://music.163.com/api/song/detail/?id={song['id']}&ids=[{song['id']}]"
        detail_response = requests.get(detail_url, headers=headers)
        detail_response.encoding = 'utf-8'
        
        if detail_response.status_code == 200:
            try:
                detail_data = detail_response.json()
                if 'songs' in detail_data and detail_data['songs']:
                    song_detail = detail_data['songs'][0]
                    publish_time = song_detail.get('publishTime', 0)
                    if publish_time > 0:
                        release_date = datetime.fromtimestamp(publish_time / 1000)
                        print(f"Song {i+1}: {song['name']} - Release: {release_date.date()}")
                    else:
                        print(f"Song {i+1}: {song['name']} - No publish time")
            except json.JSONDecodeError:
                print(f"Song {i+1}: {song['name']} - JSON error")
        
        time.sleep(0.5)
        
    except Exception as e:
        print(f"Error with song {song['id']}: {e}")

Song 1: 想你就写信 (Live) - No publish time
Song 2: 屋顶 - No publish time
Song 3: 布拉格广场 - No publish time
Song 4: 刀马旦 - No publish time
Song 5: 默 (Live) - No publish time
Song 6: 因为爱情 (Live) - No publish time
Song 7: 骑士精神 - No publish time
Song 8: 海盗 - No publish time
Song 9: 布拉格广场 - No publish time
Song 10: 刀马旦 - No publish time


## 获取每首歌的收藏数和评论数

现在对每首近两年的歌曲，获取收藏数和评论数。

In [118]:
song_data = []

for song in recent_songs:
    try:
        # 获取歌曲详情
        detail_api_url = f"https://music.163.com/api/song/detail/?id={song['id']}&ids=[{song['id']}]"
        detail_response = requests.get(detail_api_url, headers=headers)
        
        favorite_count = 0
        comment_count = 0
        
        if detail_response.status_code == 200:
            try:
                detail_data = detail_response.json()
                if 'songs' in detail_data and detail_data['songs']:
                    song_detail = detail_data['songs'][0]
                    favorite_count = song_detail.get('starredNum', 0)
                    
                    # 获取评论数
                    comment_api_url = f"https://music.163.com/api/v1/resource/comments/R_SO_4_{song['id']}?limit=1"
                    comment_response = requests.get(comment_api_url, headers=headers)
                    if comment_response.status_code == 200:
                        try:
                            comment_data = comment_response.json()
                            comment_count = comment_data.get('total', 0)
                        except:
                            pass
            except:
                pass
        
        song_data.append({
            'song_id': song['id'],
            'song_name': song['name'],
            'album': song.get('album', ''),
            'release_date': song['release_date'].date(),
            'favorite_count': favorite_count,
            'comment_count': comment_count
        })
        print(f"Processed: {song['name']} - Favorites: {favorite_count}, Comments: {comment_count}")
        
        time.sleep(1)  # 延迟
        
    except Exception as e:
        print(f"Error processing {song['name']}: {e}")
        time.sleep(2)

print(f"Collected data for {len(song_data)} songs")

Processed: 排尾后巷 - Favorites: 0, Comments: 14827
Processed: 玩转DNA - Favorites: 0, Comments: 329
Processed: 玩转 DNA (伴奏) - Favorites: 0, Comments: 3
Processed: 唤醒我 - Favorites: 0, Comments: 1238
Processed: 七号种子 - Favorites: 0, Comments: 1086
Processed: 杀死恐惧 - Favorites: 0, Comments: 1137
Processed: 所有人都要跟着唱 - Favorites: 0, Comments: 1283
Processed: 极乐鸟 - Favorites: 0, Comments: 1280
Processed: 拓金 - Favorites: 0, Comments: 1308
Processed: 前进 Marching - Favorites: 0, Comments: 6801
Processed: 平安的一刻 - Favorites: 0, Comments: 369
Processed: 明天不是世界末日 OngMa - Favorites: 0, Comments: 1611
Processed: 回家吧 Can You Hear Me - Favorites: 0, Comments: 1147
Processed: 燃月 - Favorites: 0, Comments: 302
Processed: 燃月 (伴奏) - Favorites: 0, Comments: 72
Processed: 忙忙碌碌寻宝藏 - Favorites: 0, Comments: 755
Processed: 忙忙碌碌寻宝藏 (伴奏) - Favorites: 0, Comments: 410
Processed: E妹儿 - Favorites: 0, Comments: 4731
Processed: 伍佰的风范 Come Back To Me - Favorites: 0, Comments: 1360
Processed: 没有什么是不可能的 - Favorites: 0, Comments: 

In [77]:
song_data = []

print(f"Processing {len(recent_songs)} songs...")
for i, song in enumerate(recent_songs):
    print(f"Processing song {i+1}: {song['name']} (ID: {song['id']})")

Processing 50 songs...
Processing song 1: Six Degrees (ID: 2664793877)
Processing song 2: 稻香 (Remix摇滚版) (ID: 2604902333)
Processing song 3: 床边故事 (ID: 415792916)
Processing song 4: 说走就走 (ID: 418602084)
Processing song 5: 一点点 (ID: 418603076)
Processing song 6: 前世情人 (ID: 415792918)
Processing song 7: 英雄 (ID: 418602085)
Processing song 8: 不该 (ID: 417250561)
Processing song 9: 土耳其冰淇淋 (ID: 418602086)
Processing song 10: 告白气球 (ID: 418603077)
Processing song 11: Now You See Me (ID: 417247652)
Processing song 12: 爱情废柴 (ID: 418602087)
Processing song 13: 阳明山 (ID: 29822016)
Processing song 14: 窃爱 (ID: 29822010)
Processing song 15: 算什么男人 (ID: 29818120)
Processing song 16: 天涯过客 (ID: 29822015)
Processing song 17: 怎么了 (ID: 29822033)
Processing song 18: 一口气全念对 (ID: 29822032)
Processing song 19: 我要夏天 (ID: 29822017)
Processing song 20: 手写的从前 (ID: 29822018)
Processing song 21: 鞋子特大号 (ID: 29818117)
Processing song 22: 听爸爸的话 (ID: 29822013)
Processing song 23: 美人鱼 (ID: 29822012)
Processing song 24: 听见下雨的声音 

In [39]:
# 调试：打印API响应
for song in recent_songs[:1]:  # 只测试第一首
    detail_api_url = f"https://music.163.com/api/song/detail/?id={song['id']}&ids=[{song['id']}]"
    detail_response = requests.get(detail_api_url, headers=headers)
    if detail_response.status_code == 200:
        try:
            detail_data = detail_response.json()
            print(f"API response for {song['name']}:")
            if 'songs' in detail_data and detail_data['songs']:
                song_detail = detail_data['songs'][0]
                print(f"  Keys: {list(song_detail.keys())}")
                print(f"  Sample values: {dict(list(song_detail.items())[:10])}")
        except:
            print("JSON parse error")

API response for 圣诞星 (feat. 杨瑞代):
  Keys: ['name', 'id', 'position', 'alias', 'status', 'fee', 'copyrightId', 'disc', 'no', 'artists', 'album', 'starred', 'popularity', 'score', 'starredNum', 'duration', 'playedNum', 'dayPlays', 'hearTime', 'sqMusic', 'hrMusic', 'ringtone', 'crbt', 'audition', 'copyFrom', 'commentThreadId', 'rtUrl', 'ftype', 'rtUrls', 'copyright', 'transName', 'sign', 'mark', 'originCoverType', 'originSongSimpleData', 'single', 'noCopyrightRcmd', 'hMusic', 'mMusic', 'lMusic', 'bMusic', 'mvid', 'rtype', 'rurl', 'mp3Url']
  Sample values: {'name': '圣诞星 (feat. 杨瑞代)', 'id': 2110762614, 'position': 0, 'alias': [], 'status': 0, 'fee': 0, 'copyrightId': 0, 'disc': '01', 'no': 1, 'artists': [{'name': '周杰伦', 'id': 6452, 'picId': 0, 'img1v1Id': 0, 'briefDesc': '', 'picUrl': 'https://p1.music.126.net/6y-UleORITEDbvrOLV0Q8A==/5639395138885805.jpg', 'img1v1Url': 'https://p1.music.126.net/6y-UleORITEDbvrOLV0Q8A==/5639395138885805.jpg', 'albumSize': 0, 'alias': [], 'trans': '', 'musi

## 保存数据到CSV

In [ ]:
df = pd.DataFrame(song_data)
df.to_csv('马思唯.csv', index=False, encoding='utf-8-sig')
print("Data saved to 马思唯.csv")
print(df.head())

Data saved to vinida_weng_songs_data.csv
      song_id    song_name  album release_date  favorite_count  comment_count
0  3373360561         排尾后巷   排尾后巷   2026-04-29               0          14827
1  2711424941        玩转DNA  玩转DNA   2025-06-11               0            329
2  2711424940  玩转 DNA (伴奏)  玩转DNA   2025-06-11               0              3
3  2644474582          唤醒我   七号种子   2024-11-15               0           1238
4  2644474583         七号种子   七号种子   2024-11-15               0           1086
